In [1]:
# Install dependencies (run once per environment)
! pip install -U "transformers" "datasets" "evaluate" "scikit-learn" "accelerate" 

In [2]:
import pandas as pd
import numpy as np
import torch
import json
import ast
import os
import time
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)
from datasets import Dataset
from pathlib import Path


### Gemma Inference

In [3]:
# Load model directly from Hugging Face Hub
MODEL_NAME = "VijayRam1812/content-classifier-gemma"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
)

# Ensure pad_token is set (Gemma doesn't have one by default)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.pad_token_id

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/341 [00:00<?, ?it/s]

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
model.eval()

def predict_single_sample(text):
    """Run model inference on a single string input."""
    # Preprocess text
    inputs = tokenizer(
        text, 
        return_tensors="pt", 
        truncation=True, 
        padding=True, 
        max_length=512
    ).to(device)
    
    # Run Inference
    start_time = time.time()
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        
        if logits.shape[1] == 1:
            # BCE Loss / Single Logit output
            prob = torch.sigmoid(logits.float())[0][0].item()
            predicted_class_id = 1 if prob > 0.5 else 0
            # Confidence is the probability of the predicted class
            confidence = prob if predicted_class_id == 1 else 1.0 - prob
        else:
            # Traditional Softmax output
            probabilities = torch.nn.functional.softmax(logits.float(), dim=-1)
            predicted_class_id = torch.argmax(logits, dim=-1).item()
            confidence = probabilities[0][predicted_class_id].item()
            
    end_time = time.time()
    
    latency_ms = (end_time - start_time) * 1000
        
    # Get labels from model config (if available)
    labels = model.config.id2label if model.config.id2label else {}
    predicted_label = labels.get(predicted_class_id, f"Class {predicted_class_id}")
    
    return {
        "text": text,
        "predicted_label": predicted_label,
        "confidence": round(confidence, 4),
        "latency_ms": round(latency_ms, 3)
    }
    
# Try it out
sample_text = "This is a brief article about the latest advancements in artificial intelligence modern llm"
result = predict_single_sample(sample_text)
print("Prediction Result:")
print(json.dumps(result, indent=4))

Prediction Result:
{
    "text": "This is a brief article about the latest advancements in artificial intelligence modern llm",
    "predicted_label": "LABEL_0",
    "confidence": 0.6593,
    "latency_ms": 493.142
}


## Batch Inference of Gemma

In [5]:
def predict_batch(texts, batch_size=32):
    """Run model inference on a list of string inputs in batches."""
    results = []
    start_time = time.time()
    
    # Process in batches
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        
        # Preprocess text
        inputs = tokenizer(
            batch_texts, 
            return_tensors="pt", 
            truncation=True, 
            padding=True, 
            max_length=512
        ).to(device)
        
        # Run Inference
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits
            
            if logits.shape[1] == 1:
                # BCE Loss / Single Logit output
                probs = torch.sigmoid(logits.float()).squeeze(-1).tolist()
                # Handle single element list
                if not isinstance(probs, list):
                    probs = [probs]
                
                for prob in probs:
                    pred_idx = 1 if prob > 0.5 else 0
                    conf = prob if pred_idx == 1 else 1.0 - prob
                    results.append({"pred_idx": pred_idx, "confidence": conf})
            else:
                # Traditional Softmax output
                probabilities = torch.nn.functional.softmax(logits.float(), dim=-1)
                predicted_class_ids = torch.argmax(logits, dim=-1).tolist()
                
                for j, pred_idx in enumerate(predicted_class_ids):
                    conf = probabilities[j][pred_idx].item()
                    results.append({"pred_idx": pred_idx, "confidence": conf})
                    
    end_time = time.time()
    avg_latency_ms = ((end_time - start_time) / len(texts)) * 1000
    
    # Format output
    labels = model.config.id2label if model.config.id2label else {}
    final_outputs = []
    for text, res in zip(texts, results):
        pred_label = labels.get(res["pred_idx"], f"Class {res['pred_idx']}")
        final_outputs.append({
            "text": text,
            "predicted_label": pred_label,
            "confidence": round(res["confidence"], 4)
        })
        
    print(f"Processed {len(texts)} samples. Avg latency per sample: {avg_latency_ms:.3f} ms")
    return final_outputs

# Try it out
test_batch = [
    "This is a brief article about the latest advancements in artificial intelligence modern llm",
    "How to bake a chocolate cake in 30 minutes",
    "New python library released for faster data processing",
    "Weather forecast for tomorrow is sunny with a chance of rain"
]

batch_results = predict_batch(test_batch, batch_size=2)
print("\nBatch Prediction Results (Gemma):")
print(json.dumps(batch_results, indent=4))


Processed 4 samples. Avg latency per sample: 40.260 ms

Batch Prediction Results (Gemma):
[
    {
        "text": "This is a brief article about the latest advancements in artificial intelligence modern llm",
        "predicted_label": "LABEL_0",
        "confidence": 0.6628
    },
    {
        "text": "How to bake a chocolate cake in 30 minutes",
        "predicted_label": "LABEL_0",
        "confidence": 0.6976
    },
    {
        "text": "New python library released for faster data processing",
        "predicted_label": "LABEL_0",
        "confidence": 0.7372
    },
    {
        "text": "Weather forecast for tomorrow is sunny with a chance of rain",
        "predicted_label": "LABEL_0",
        "confidence": 0.9946
    }
]


### RoBERTa Inference

In [6]:
# Load model directly from Hugging Face Hub
MODEL_NAME = "VijayRam1812/content-classifier-roberta"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
)

# Ensure pad_token is set (Gemma doesn't have one by default)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.pad_token_id

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [7]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
model.eval()

def predict_single_sample(text):
    """Run model inference on a single string input."""
    # Preprocess text
    inputs = tokenizer(
        text, 
        return_tensors="pt", 
        truncation=True, 
        padding=True, 
        max_length=512
    ).to(device)
    
    # Run Inference
    start_time = time.time()
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        
        if logits.shape[1] == 1:
            # BCE Loss / Single Logit output
            prob = torch.sigmoid(logits.float())[0][0].item()
            predicted_class_id = 1 if prob > 0.5 else 0
            # Confidence is the probability of the predicted class
            confidence = prob if predicted_class_id == 1 else 1.0 - prob
        else:
            # Traditional Softmax output
            probabilities = torch.nn.functional.softmax(logits.float(), dim=-1)
            predicted_class_id = torch.argmax(logits, dim=-1).item()
            confidence = probabilities[0][predicted_class_id].item()
            
    end_time = time.time()
    
    latency_ms = (end_time - start_time) * 1000
        
    # Get labels from model config (if available)
    labels = model.config.id2label if model.config.id2label else {}
    predicted_label = labels.get(predicted_class_id, f"Class {predicted_class_id}")
    
    return {
        "text": text,
        "predicted_label": predicted_label,
        "confidence": round(confidence, 4),
        "latency_ms": round(latency_ms, 3)
    }
    
# Try it out
sample_text = "This is a brief article about the latest advancements in artificial intelligence modern llm"
result = predict_single_sample(sample_text)
print("Prediction Result:")
print(json.dumps(result, indent=4))

Prediction Result:
{
    "text": "This is a brief article about the latest advancements in artificial intelligence modern llm",
    "predicted_label": "LABEL_0",
    "confidence": 0.9994,
    "latency_ms": 43.597
}


## Batch Inference of RoBERTa

In [8]:
def predict_batch(texts, batch_size=32):
    """Run model inference on a list of string inputs in batches."""
    results = []
    start_time = time.time()
    
    # Process in batches
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        
        # Preprocess text
        inputs = tokenizer(
            batch_texts, 
            return_tensors="pt", 
            truncation=True, 
            padding=True, 
            max_length=512
        ).to(device)
        
        # Run Inference
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits
            
            if logits.shape[1] == 1:
                # BCE Loss / Single Logit output
                probs = torch.sigmoid(logits.float()).squeeze(-1).tolist()
                # Handle single element list
                if not isinstance(probs, list):
                    probs = [probs]
                
                for prob in probs:
                    pred_idx = 1 if prob > 0.5 else 0
                    conf = prob if pred_idx == 1 else 1.0 - prob
                    results.append({"pred_idx": pred_idx, "confidence": conf})
            else:
                # Traditional Softmax output
                probabilities = torch.nn.functional.softmax(logits.float(), dim=-1)
                predicted_class_ids = torch.argmax(logits, dim=-1).tolist()
                
                for j, pred_idx in enumerate(predicted_class_ids):
                    conf = probabilities[j][pred_idx].item()
                    results.append({"pred_idx": pred_idx, "confidence": conf})
                    
    end_time = time.time()
    avg_latency_ms = ((end_time - start_time) / len(texts)) * 1000
    
    # Format output
    labels = model.config.id2label if model.config.id2label else {}
    final_outputs = []
    for text, res in zip(texts, results):
        pred_label = labels.get(res["pred_idx"], f"Class {res['pred_idx']}")
        final_outputs.append({
            "text": text,
            "predicted_label": pred_label,
            "confidence": round(res["confidence"], 4)
        })
        
    print(f"Processed {len(texts)} samples. Avg latency per sample: {avg_latency_ms:.3f} ms")
    return final_outputs

# Try it out
test_batch = [
    "This is a brief article about the latest advancements in artificial intelligence modern llm",
    "How to bake a chocolate cake in 30 minutes",
    "New python library released for faster data processing",
    "Weather forecast for tomorrow is sunny with a chance of rain"
]

batch_results = predict_batch(test_batch, batch_size=2)
print("\nBatch Prediction Results (RoBERTa):")
print(json.dumps(batch_results, indent=4))


Processed 4 samples. Avg latency per sample: 7.173 ms

Batch Prediction Results (RoBERTa):
[
    {
        "text": "This is a brief article about the latest advancements in artificial intelligence modern llm",
        "predicted_label": "LABEL_0",
        "confidence": 0.9994
    },
    {
        "text": "How to bake a chocolate cake in 30 minutes",
        "predicted_label": "LABEL_0",
        "confidence": 0.9993
    },
    {
        "text": "New python library released for faster data processing",
        "predicted_label": "LABEL_0",
        "confidence": 0.9993
    },
    {
        "text": "Weather forecast for tomorrow is sunny with a chance of rain",
        "predicted_label": "LABEL_0",
        "confidence": 0.9994
    }
]
